# Notebook 01 — Coleta (Camada Bronze)

**MVP de Engenharia de Dados** · PUC-Rio · Sprint 3

---

### Objetivo

Realizar a ingestão dos dados brutos do portal de Dados Abertos do ONS para o ambiente de nuvem, preservando os arquivos exatamente como disponibilizados pela fonte.

### Fonte

- **Dataset:** Balanço de Energia nos Subsistemas (base horária)
- **Portal:** https://dados.ons.org.br/dataset/balanco-energia-subsistema
- **Licença:** Creative Commons Attribution (CC-BY) — uso livre mediante atribuição ao ONS
- **Recorte:** 2019 a 2025 (7 anos completos)

### Entrada e saída

| | |
|---|---|
| **Entrada** | URLs públicas do bucket do ONS na AWS |
| **Saída** | 7 arquivos CSV em `/Volumes/workspace/bronze/raw_ons/` |

### Por que a coleta é automatizada

O download é feito por código, não por upload manual. Isso torna a coleta **reproduzível** (qualquer pessoa executa e obtém o mesmo resultado), **auditável** (o código versionado é a documentação do processo) e **escalável** (incluir um novo ano é alterar um número).

## 1. Download dos arquivos para o Volume

O laço percorre cada ano, baixa o CSV correspondente e grava no Volume do Unity Catalog.

Dois cuidados importantes:

- **Idempotência:** arquivos já presentes são ignorados, permitindo reexecutar o notebook sem retrabalho. Pipelines são reexecutados com frequência na prática.
- **`raise_for_status()`:** faz o código falhar explicitamente se a URL retornar erro, em vez de gravar um arquivo corrompido silenciosamente.

In [0]:
import os, requests

ANOS = list(range(2019, 2026))
URL_BASE = "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/balanco_energia_subsistema_ho"
VOLUME_BRONZE = "/Volumes/workspace/bronze/raw_ons"

for ano in ANOS:
    nome = f"BALANCO_ENERGIA_SUBSISTEMA_{ano}.csv"
    destino = f"{VOLUME_BRONZE}/{nome}"
    if os.path.exists(destino):
        print(f"[JA EXISTE] {nome}")
        continue
    print(f"[BAIXANDO ] {nome}")
    r = requests.get(f"{URL_BASE}/{nome}", timeout=600)
    r.raise_for_status()
    with open(destino, "wb") as f:
        f.write(r.content)
    print(f"[OK       ] {nome} ({len(r.content)/1024/1024:.2f} MB)")

print("\nIngestão concluída.")

[JA EXISTE] BALANCO_ENERGIA_SUBSISTEMA_2019.csv
[JA EXISTE] BALANCO_ENERGIA_SUBSISTEMA_2020.csv
[JA EXISTE] BALANCO_ENERGIA_SUBSISTEMA_2021.csv
[JA EXISTE] BALANCO_ENERGIA_SUBSISTEMA_2022.csv
[JA EXISTE] BALANCO_ENERGIA_SUBSISTEMA_2023.csv
[JA EXISTE] BALANCO_ENERGIA_SUBSISTEMA_2024.csv
[JA EXISTE] BALANCO_ENERGIA_SUBSISTEMA_2025.csv

Ingestão concluída.


## 2. Conferência — arquivos persistidos no Volume

Evidência de que os arquivos foram efetivamente gravados na nuvem.

A listagem mostra oito arquivos: os sete do Balanço de Energia e o
`CAPACIDADE_GERACAO.csv`, baixado pelo notebook 07 para o mesmo Volume. Por isso a
leitura do notebook 02 filtra por prefixo de nome, e não por extensão — ler
`*.csv` num Volume compartilhado carregaria o arquivo errado junto, sem gerar erro.

In [0]:
for a in sorted(dbutils.fs.ls(VOLUME_BRONZE), key=lambda f: f.name):
    print(f"{a.name:45s} {a.size/1024/1024:8.2f} MB")

BALANCO_ENERGIA_SUBSISTEMA_2019.csv               4.70 MB
BALANCO_ENERGIA_SUBSISTEMA_2020.csv               4.70 MB
BALANCO_ENERGIA_SUBSISTEMA_2021.csv               4.86 MB
BALANCO_ENERGIA_SUBSISTEMA_2022.csv               4.89 MB
BALANCO_ENERGIA_SUBSISTEMA_2023.csv               4.32 MB
BALANCO_ENERGIA_SUBSISTEMA_2024.csv               4.36 MB
BALANCO_ENERGIA_SUBSISTEMA_2025.csv               3.89 MB
CAPACIDADE_GERACAO.csv                            1.22 MB


## 3. Inspeção do formato bruto

Antes de ler o CSV com o Spark, é preciso descobrir como ele realmente é: qual o separador, qual a codificação, se tem cabeçalho.

**Nunca assuma o formato de um arquivo — abra e olhe.** A página do dataset informa que os arquivos são delimitados por vírgula; a inspeção mostrou que o delimitador real é **ponto e vírgula**. Se a documentação tivesse sido aceita sem verificação, o Spark carregaria todo o conteúdo em uma única coluna.

A saída revela também que zeros da geração fotovoltaica aparecem na origem em notação científica (`0E-8`), tratado adiante na camada Silver.

In [0]:
with open(f"{VOLUME_BRONZE}/BALANCO_ENERGIA_SUBSISTEMA_2019.csv", "r", encoding="utf-8") as f:
    for i in range(3):
        print(f"Linha {i}: {f.readline().rstrip()}")

Linha 0: id_subsistema;nom_subsistema;din_instante;val_gerhidraulica;val_gertermica;val_gereolica;val_gersolar;val_carga;val_intercambio
Linha 1: NE;NORDESTE;2019-01-01 00:00:00;2292.41700000;873.48200000;5320.80899999;0E-8;9831.71799999;-1345.01000000
Linha 2: N;NORTE;2019-01-01 00:00:00;7297.07300000;1416.71899999;142.23700000;0E-8;4888.03300000;3967.99600000
